# Policy comparison study for CI diagnosis

This notebook implements the four operating policies described in the decision record. We keep the model explicit and traceable:
- priors, likelihoods, and costs live in `experiments/constants.py`
- all policies are evaluated on the same benchmark cases
- every policy is reported with action-wise metrics, cost summaries, and human-involvement rates

## Assumptions

1. The hidden state is the true CI root-cause category (`S1`..`S7`).
2. The benchmark uses the generated benchmark rows in the project dataset.
3. Evidence order is `E1 -> E2 -> E3 -> E4` and the free evidence sources are `E1` and `E2`.
4. The action map follows the project decision policy:
   - `S1`, `S2`, `S4` -> `Fix Lint`
   - `S3` -> `Fix Dependency`
   - `S5`, `S6`, `S7` -> `Escalate`
5. Decision cost is drawn from the action-by-state matrix in `decisions/costs.md`.
6. Information costs follow the same matrix: `E1 = $0.00`, `E2 = $0.00`, `E3 = $0.07`, `E4 = $33.33`.
7. For the threshold-based policies, the 'uncertain band' is defined as the region where the best expected-cost action is not materially better than the next best action or the posterior max is too low to act confidently.
8. For the value-of-information policy, we rank remaining paid checks by `EIG / cost` and stop once the remaining gain is too small to justify a paid check.


In [1]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

candidate_roots = [
    Path.cwd(),
    Path('/home/divas/ml/CI-diagnosis-agent'),
    Path('/home/divas/ml/CI-diagnosis-agent.worktrees/cost-analysis-and-reporting-v0-v1'),
]
project_root = next((r for r in candidate_roots if (r / 'experiments').exists() and (r / 'experiments' / 'constants.py').exists()), Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from experiments.constants import (
    ACTION_COSTS,
    ACTION_LABELS,
    ACTION_ORDER,
    EVIDENCE_COSTS,
    EVIDENCE_ORDER,
    LIKELIHOODS,
    PRIORS,
    STATES,
    STATE_TO_ACTION,
    bayes_update,
)

bench_path = project_root / 'data' / 'benchmark_data' / 'final_test_cases.jsonl'
if not bench_path.exists():
    alt = Path('/home/divas/ml/CI-diagnosis-agent/data/benchmark_data/final_test_cases.jsonl')
    if alt.exists():
        bench_path = alt
print(f'Loaded benchmark rows from: {bench_path}')
rows = [json.loads(line) for line in bench_path.read_text().splitlines() if line.strip()]
print(f'Loaded benchmark rows: {len(rows)}')


Loaded benchmark rows from: /home/divas/ml/CI-diagnosis-agent/data/benchmark_data/final_test_cases.jsonl
Loaded benchmark rows: 153


In [1]:
def flatten_row(row):
    gt = row.get('ground_truth') or {}
    ev = row.get('evidence') or {}
    soa = row.get('scenario_optimal_action') or {}
    return {
        'test_id': row.get('test_id'),
        'split': row.get('split'),
        'ground_truth': gt.get('state') if isinstance(gt, dict) else None,
        'E1_outcome': ev.get('E1', {}).get('outcome') if isinstance(ev, dict) else None,
        'E2_outcome': ev.get('E2', {}).get('outcome') if isinstance(ev, dict) else None,
        'E3_outcome': ev.get('E3', {}).get('outcome') if isinstance(ev, dict) else None,
        'E4_outcome': ev.get('E4', {}).get('outcome') if isinstance(ev, dict) else None,
        'ground_action': soa.get('ground_action') if isinstance(soa, dict) else None,
        'optimal_action': soa.get('optimal_action') if isinstance(soa, dict) else None,
    }

df = pd.DataFrame([flatten_row(r) for r in rows])
print(df.head(3).to_string(index=False))
print(f'Rows with ground_action: {df['ground_action'].notna().sum()} / {len(df)}')


    test_id  split ground_truth E1_outcome E2_outcome    E3_outcome               E4_outcome  ground_action     optimal_action
easy_000001 medium           S4          A        src pass_on_rerun not_reproducible_locally       Fix Lint           fix_code
easy_000002   easy           S1          C        src fail_on_rerun     reproducible_locally       Escalate           fix_code
easy_000003   easy           S3          A     config fail_on_rerun not_reproducible_locally Fix Dependency resolve_dependency
Rows with ground_action: 152 / 153


In [1]:
def action_cost_for_state(action, state):
    return ACTION_COSTS.get(action, {}).get(state, ACTION_COSTS['Escalate'][state])

def state_to_action(state):
    return STATE_TO_ACTION.get(state, 'Escalate')

def expected_action_cost(posterior, action):
    return sum(posterior[s] * ACTION_COSTS[action][s] for s in STATES)

def choose_expected_cost_action(posterior):
    costs = {a: expected_action_cost(posterior, a) for a in ACTION_ORDER}
    ordered = sorted(costs.items(), key=lambda kv: kv[1])
    best_action, best_cost = ordered[0]
    second_cost = ordered[1][1]
    if (second_cost - best_cost < 5.0) or (max(posterior.values()) < 0.55):
        return 'Escalate', costs
    return best_action, costs

def entropy(prob_dict):
    ps = [p for p in prob_dict.values() if p > 0]
    return -sum(p * math.log2(p) for p in ps)

def expected_information_gain(posterior, evidence_key):
    outcomes = sorted({outcome for state in STATES for outcome in LIKELIHOODS[evidence_key][state].keys()})
    expected_conditional_entropy = 0.0
    for outcome in outcomes:
        p_outcome = sum(posterior[state] * LIKELIHOODS[evidence_key][state].get(outcome, 0.0) for state in STATES)
        if p_outcome <= 0:
            continue
        posterior_after = {}
        for state in STATES:
            likelihood = LIKELIHOODS[evidence_key][state].get(outcome, 0.0)
            posterior_after[state] = (posterior[state] * likelihood) / p_outcome
        expected_conditional_entropy += p_outcome * entropy(posterior_after)
    return entropy(posterior) - expected_conditional_entropy

def print_policy_metrics(y_true, y_pred, label):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', labels=ACTION_LABELS, zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', labels=ACTION_LABELS, zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', labels=ACTION_LABELS, zero_division=0)
    print(f'=== {label} ===')
    print(f'Accuracy:  {accuracy:.4f}')
    print(f'Precision (macro): {precision:.4f}')
    print(f'Recall (macro):    {recall:.4f}')
    print(f'F1 (macro):       {f1:.4f}')
    print()
    print(classification_report(y_true, y_pred, labels=ACTION_LABELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=ACTION_LABELS)
    print('Confusion matrix (rows=true, cols=pred):')
    print(pd.DataFrame(cm, index=ACTION_LABELS, columns=ACTION_LABELS))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ACTION_LABELS)
    disp.plot(cmap='Blues', xticks_rotation=30)
    plt.title(f'{label} confusion matrix')
    plt.tight_layout()
    plt.show()
    print()
    return {'accuracy': accuracy, 'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1}

def build_policy_summary(name, y_true, y_pred, info_costs, decision_costs):
    metrics = print_policy_metrics(y_true, y_pred, name)
    total_info = sum(info_costs)
    total_decision = sum(decision_costs)
    total_cost = total_info + total_decision
    human_pct = (pd.Series(y_pred) == 'Escalate').mean() * 100
    print('Cost summary:')
    print(f'Total decision cost: ${total_decision:,.2f}')
    print(f'Total information cost: ${total_info:,.2f}')
    print(f'Total cost: ${total_cost:,.2f}')
    print(f'Expected cost / case: ${total_cost / len(y_pred):,.2f}')
    print(f'Human escalation percentage: {human_pct:.2f}%')
    print()
    metrics.update({
        'total_decision_cost': total_decision,
        'total_information_cost': total_info,
        'total_cost': total_cost,
        'expected_cost_per_case': total_cost / len(y_pred),
        'human_pct': human_pct,
    })
    return metrics

print('Helper functions defined.')


Helper functions defined.


## P0 — Baseline policy

This baseline does not look at evidence. It always predicts the most common action label in the benchmark so that we can prove the more advanced policies beat doing nothing.


In [1]:
eval_df = df.dropna(subset=['ground_action']).copy()
majority_action = eval_df['ground_action'].value_counts().idxmax()
eval_df['p0_prediction'] = majority_action
y_true_p0 = eval_df['ground_action']
y_pred_p0 = eval_df['p0_prediction']
p0_decision_costs = [ACTION_COSTS[majority_action][state] for state in eval_df['ground_truth']]
p0_info_costs = [0.0 for _ in range(len(eval_df))]
p0_metrics = build_policy_summary('P0 — baseline majority action', y_true_p0, y_pred_p0, p0_info_costs, p0_decision_costs)
p0_summary = {'policy': 'P0', 'metrics': p0_metrics, 'predictions': y_pred_p0.to_list()}


=== P0 — baseline majority action ===
Accuracy:  0.5000
Precision (macro): 0.1667
Recall (macro):    0.3333
F1 (macro):       0.2222

                precision    recall  f1-score   support

      Escalate       0.50      1.00      0.67        76
Fix Dependency       0.00      0.00      0.00        45
      Fix Lint       0.00      0.00      0.00        31

      accuracy                           0.50       152
     macro avg       0.17      0.33      0.22       152
  weighted avg       0.25      0.50      0.33       152

Confusion matrix (rows=true, cols=pred):
                Escalate  Fix Dependency  Fix Lint
Escalate              76               0         0
Fix Dependency        45               0         0
Fix Lint              31               0         0

Cost summary:
Total decision cost: $7,600.00
Total information cost: $0.00
Total cost: $7,600.00
Expected cost / case: $50.00
Human escalation percentage: 100.00%



## P1 — Belief-only policy

This policy updates `E1` and `E2` for free, then chooses the highest-posterior state and maps that hidden state back to an action. There is no cost-aware thresholding here.


In [ ]:
def run_policy_p1(row):
    posterior = PRIORS.copy()
    used = []
    for evidence_key in ['E1', 'E2']:
        outcome = row.get(f'{evidence_key}_outcome')
        if outcome is None:
            continue
        posterior = bayes_update(posterior, evidence_key, outcome)
        used.append(evidence_key)
    best_state = max(posterior, key=posterior.get)
    action = state_to_action(best_state)
    info_cost = sum(EVIDENCE_COSTS[k] for k in used)
    decision_cost = ACTION_COSTS[action][row['ground_truth']]
    return action, info_cost, decision_cost, used

eval_df = df.dropna(subset=['ground_action']).copy()
p1_records = []
for _, row in eval_df.iterrows():
    action, info_cost, decision_cost, used = run_policy_p1(row)
    p1_records.append({'ground_truth': row['ground_truth'], 'prediction': action, 'info_cost': info_cost, 'decision_cost': decision_cost, 'used': used})
p1_df = pd.DataFrame(p1_records)
y_true_p1 = eval_df['ground_action']
y_pred_p1 = p1_df['prediction']
p1_metrics = build_policy_summary('P1 — belief-only policy', y_true_p1, y_pred_p1, p1_df['info_cost'].tolist(), p1_df['decision_cost'].tolist())
p1_summary = {'policy': 'P1', 'metrics': p1_metrics, 'predictions': y_pred_p1.to_list()}


=== P1 — belief-only policy ===
Accuracy:  0.6184
Precision (macro): 0.6943
Recall (macro):    0.6983
F1 (macro):       0.6284

                precision    recall  f1-score   support

      Escalate       0.93      0.36      0.51        76
Fix Dependency       0.49      0.93      0.65        45
      Fix Lint       0.66      0.81      0.72        31

      accuracy                           0.62       152
     macro avg       0.69      0.70      0.63       152
  weighted avg       0.75      0.62      0.60       152

Confusion matrix (rows=true, cols=pred):
                Escalate  Fix Dependency  Fix Lint
Escalate              27              39        10
Fix Dependency         0              42         3
Fix Lint               2               4        25

Cost summary:
Total decision cost: $6,812.69
Total information cost: $0.00
Total cost: $6,812.69
Expected cost / case: $44.82
Human escalation percentage: 19.08%



## P2 — Threshold policy

This policy still uses only free evidence, but it selects the action with minimum expected cost under the current posterior. If the decision is in the uncertain band, it escalates rather than acting.


In [1]:
def run_policy_p2(row):
    posterior = PRIORS.copy()
    used = []
    for evidence_key in ['E1', 'E2']:
        outcome = row.get(f'{evidence_key}_outcome')
        if outcome is None:
            continue
        posterior = bayes_update(posterior, evidence_key, outcome)
        used.append(evidence_key)
    action, costs = choose_expected_cost_action(posterior)
    info_cost = sum(EVIDENCE_COSTS[k] for k in used)
    decision_cost = ACTION_COSTS[action][row['ground_truth']]
    return action, info_cost, decision_cost, used, costs

eval_df = df.dropna(subset=['ground_action']).copy()
p2_records = []
for _, row in eval_df.iterrows():
    action, info_cost, decision_cost, used, costs = run_policy_p2(row)
    p2_records.append({'ground_truth': row['ground_truth'], 'prediction': action, 'info_cost': info_cost, 'decision_cost': decision_cost, 'used': used, 'costs': costs})
p2_df = pd.DataFrame(p2_records)
y_true_p2 = eval_df['ground_action']
y_pred_p2 = p2_df['prediction']
p2_metrics = build_policy_summary('P2 — expected-cost threshold policy', y_true_p2, y_pred_p2, p2_df['info_cost'].tolist(), p2_df['decision_cost'].tolist())
p2_summary = {'policy': 'P2', 'metrics': p2_metrics, 'predictions': y_pred_p2.to_list()}


=== P2 — expected-cost threshold policy ===
Accuracy:  0.5987
Precision (macro): 0.6555
Recall (macro):    0.5443
F1 (macro):       0.5611

                precision    recall  f1-score   support

      Escalate       0.57      0.84      0.68        76
Fix Dependency       0.40      0.18      0.25        45
      Fix Lint       1.00      0.61      0.76        31

      accuracy                           0.60       152
     macro avg       0.66      0.54      0.56       152
  weighted avg       0.61      0.60      0.57       152

Confusion matrix (rows=true, cols=pred):
                Escalate  Fix Dependency  Fix Lint
Escalate              64              12         0
Fix Dependency        37               8         0
Fix Lint              12               0        19

Cost summary:
Total decision cost: $6,775.75
Total information cost: $0.00
Total cost: $6,775.75
Expected cost / case: $44.58
Human escalation percentage: 74.34%



## P3 — Value-of-information policy

This policy begins with free evidence and then selects paid checks by highest expected information gain per unit cost. It continues only while the remaining `EIG / cost` is worth the extra check.


In [1]:
def run_policy_p3(row):
    posterior = PRIORS.copy()
    used = []
    for evidence_key in ['E1', 'E2']:
        outcome = row.get(f'{evidence_key}_outcome')
        if outcome is None:
            continue
        posterior = bayes_update(posterior, evidence_key, outcome)
        used.append(evidence_key)

    while True:
        pending = [k for k in ['E3', 'E4'] if k not in used and row.get(f'{k}_outcome') is not None]
        if not pending:
            break
        ranked = []
        for evidence_key in pending:
            ig = expected_information_gain(posterior, evidence_key)
            score = ig / EVIDENCE_COSTS[evidence_key]
            ranked.append((score, evidence_key, ig))
        ranked.sort(reverse=True)
        best_score, best_key, best_ig = ranked[0]
        if best_score <= 0.05:
            break
        outcome = row.get(f'{best_key}_outcome')
        if outcome is None:
            break
        posterior = bayes_update(posterior, best_key, outcome)
        used.append(best_key)
        action, _ = choose_expected_cost_action(posterior)
        if action != 'Escalate' and max(posterior.values()) >= 0.55:
            break

    action, _ = choose_expected_cost_action(posterior)
    info_cost = sum(EVIDENCE_COSTS[k] for k in used)
    decision_cost = ACTION_COSTS[action][row['ground_truth']]
    return action, info_cost, decision_cost, used

eval_df = df.dropna(subset=['ground_action']).copy()
p3_records = []
for _, row in eval_df.iterrows():
    action, info_cost, decision_cost, used = run_policy_p3(row)
    p3_records.append({'ground_truth': row['ground_truth'], 'prediction': action, 'info_cost': info_cost, 'decision_cost': decision_cost, 'used': used})
p3_df = pd.DataFrame(p3_records)
y_true_p3 = eval_df['ground_action']
y_pred_p3 = p3_df['prediction']
p3_metrics = build_policy_summary('P3 — value-of-information policy', y_true_p3, y_pred_p3, p3_df['info_cost'].tolist(), p3_df['decision_cost'].tolist())
p3_summary = {'policy': 'P3', 'metrics': p3_metrics, 'predictions': y_pred_p3.to_list()}


=== P3 — value-of-information policy ===
Accuracy:  0.6184
Precision (macro): 0.6560
Recall (macro):    0.5765
F1 (macro):       0.5882

                precision    recall  f1-score   support

      Escalate       0.59      0.84      0.69        76
Fix Dependency       0.38      0.18      0.24        45
      Fix Lint       1.00      0.71      0.83        31

      accuracy                           0.62       152
     macro avg       0.66      0.58      0.59       152
  weighted avg       0.61      0.62      0.59       152

Confusion matrix (rows=true, cols=pred):
                Escalate  Fix Dependency  Fix Lint
Escalate              64              12         0
Fix Dependency        37               8         0
Fix Lint               8               1        22

Cost summary:
Total decision cost: $6,675.81
Total information cost: $10.64
Total cost: $6,686.45
Expected cost / case: $43.99
Human escalation percentage: 71.71%



## P4 — decision-theoretic VOI policy

This policy is a true cost-sensitive Value of Information decision. It compares the expected loss of acting now with the expected loss after each possible outcome of each remaining evidence source, including the evidence cost. This is not entropy-based Expected Information Gain; it is decision-theoretic VOI based on downstream action costs.


In [ ]:
def run_policy_p4(row):
    posterior = PRIORS.copy()
    used = []
    for evidence_key in ['E1', 'E2']:
        outcome = row.get(f'{evidence_key}_outcome')
        if outcome is None:
            continue
        posterior = bayes_update(posterior, evidence_key, outcome)
        used.append(evidence_key)

    while True:
        pending = [k for k in ['E3', 'E4'] if k not in used and row.get(f'{k}_outcome') is not None]
        if not pending:
            break

        # This is decision-theoretic VOI, not entropy-based Expected Information Gain.
        # We compare the expected loss from acting now to the expected loss after acquiring
        # each evidence source and then acting optimally under each outcome posterior.
        current_action, current_costs = choose_expected_cost_action(posterior)
        current_expected_loss = min(current_costs.values())
        best_voi = None
        best_key = None
        for evidence_key in pending:
            outcomes = sorted({outcome for state in STATES for outcome in LIKELIHOODS[evidence_key][state].keys()})
            expected_loss_after_evidence = 0.0
            for outcome in outcomes:
                p_outcome = sum(posterior[state] * LIKELIHOODS[evidence_key][state].get(outcome, 0.0) for state in STATES)
                if p_outcome <= 0:
                    continue
                post = {}
                for state in STATES:
                    likelihood = LIKELIHOODS[evidence_key][state].get(outcome, 0.0)
                    post[state] = (posterior[state] * likelihood) / p_outcome
                action_i, costs_i = choose_expected_cost_action(post)
                outcome_loss = min(costs_i.values())
                expected_loss_after_evidence += p_outcome * outcome_loss
            expected_loss_after_evidence += EVIDENCE_COSTS[evidence_key]
            voi = current_expected_loss - expected_loss_after_evidence
            if best_voi is None or voi > best_voi:
                best_voi = voi
                best_key = evidence_key

        if best_key is None or best_voi <= 0:
            break

        outcome = row.get(f'{best_key}_outcome')
        if outcome is None:
            break
        posterior = bayes_update(posterior, best_key, outcome)
        used.append(best_key)

    action, _ = choose_expected_cost_action(posterior)
    info_cost = sum(EVIDENCE_COSTS[k] for k in used)
    decision_cost = ACTION_COSTS[action][row['ground_truth']]
    return action, info_cost, decision_cost, used

eval_df = df.dropna(subset=['ground_action']).copy()
p4_records = []
for _, row in eval_df.iterrows():
    action, info_cost, decision_cost, used = run_policy_p4(row)
    p4_records.append({'ground_truth': row['ground_truth'], 'prediction': action, 'info_cost': info_cost, 'decision_cost': decision_cost, 'used': used})
p4_df = pd.DataFrame(p4_records)
y_true_p4 = eval_df['ground_action']
y_pred_p4 = p4_df['prediction']
p4_metrics = build_policy_summary('P4 — decision-theoretic VOI policy', y_true_p4, y_pred_p4, p4_df['info_cost'].tolist(), p4_df['decision_cost'].tolist())
p4_summary = {'policy': 'P4', 'metrics': p4_metrics, 'predictions': y_pred_p4.to_list()}


In [1]:
summary_rows = [
    {
        'policy': 'P0',
        'total_decision_cost': p0_metrics['total_decision_cost'],
        'total_information_cost': p0_metrics['total_information_cost'],
        'total_cost': p0_metrics['total_cost'],
        'expected_cost_per_case': p0_metrics['expected_cost_per_case'],
        'human_pct': p0_metrics['human_pct'],
        'accuracy': p0_metrics['accuracy'],
        'precision_macro': p0_metrics['precision_macro'],
        'recall_macro': p0_metrics['recall_macro'],
        'f1_macro': p0_metrics['f1_macro'],
    },
    {
        'policy': 'P1',
        'total_decision_cost': p1_metrics['total_decision_cost'],
        'total_information_cost': p1_metrics['total_information_cost'],
        'total_cost': p1_metrics['total_cost'],
        'expected_cost_per_case': p1_metrics['expected_cost_per_case'],
        'human_pct': p1_metrics['human_pct'],
        'accuracy': p1_metrics['accuracy'],
        'precision_macro': p1_metrics['precision_macro'],
        'recall_macro': p1_metrics['recall_macro'],
        'f1_macro': p1_metrics['f1_macro'],
    },
    {
        'policy': 'P2',
        'total_decision_cost': p2_metrics['total_decision_cost'],
        'total_information_cost': p2_metrics['total_information_cost'],
        'total_cost': p2_metrics['total_cost'],
        'expected_cost_per_case': p2_metrics['expected_cost_per_case'],
        'human_pct': p2_metrics['human_pct'],
        'accuracy': p2_metrics['accuracy'],
        'precision_macro': p2_metrics['precision_macro'],
        'recall_macro': p2_metrics['recall_macro'],
        'f1_macro': p2_metrics['f1_macro'],
    },
    {
        'policy': 'P3',
        'total_decision_cost': p3_metrics['total_decision_cost'],
        'total_information_cost': p3_metrics['total_information_cost'],
        'total_cost': p3_metrics['total_cost'],
        'expected_cost_per_case': p3_metrics['expected_cost_per_case'],
        'human_pct': p3_metrics['human_pct'],
        'accuracy': p3_metrics['accuracy'],
        'precision_macro': p3_metrics['precision_macro'],
        'recall_macro': p3_metrics['recall_macro'],
        'f1_macro': p3_metrics['f1_macro'],
    },
    {
        'policy': 'P4',
        'total_decision_cost': p4_metrics['total_decision_cost'],
        'total_information_cost': p4_metrics['total_information_cost'],
        'total_cost': p4_metrics['total_cost'],
        'expected_cost_per_case': p4_metrics['expected_cost_per_case'],
        'human_pct': p4_metrics['human_pct'],
        'accuracy': p4_metrics['accuracy'],
        'precision_macro': p4_metrics['precision_macro'],
        'recall_macro': p4_metrics['recall_macro'],
        'f1_macro': p4_metrics['f1_macro'],
    },
]
summary_df = pd.DataFrame(summary_rows)
print(summary_df[['policy', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'total_decision_cost', 'total_information_cost', 'total_cost', 'expected_cost_per_case', 'human_pct']].to_string(index=False))


policy  accuracy  precision_macro  recall_macro  f1_macro  total_decision_cost  total_information_cost  total_cost  expected_cost_per_case  human_pct
    P0  0.500000         0.166667      0.333333  0.222222              7600.00                    0.00     7600.00               50.000000 100.000000
    P1  0.618421         0.694349      0.698349  0.628359              6812.69                    0.00     6812.69               44.820329  19.078947
    P2  0.598684         0.655457      0.544262  0.561134              6775.75                    0.00     6775.75               44.577303  74.342105
    P3  0.618421         0.656036      0.576520  0.588168              6675.81                   10.64     6686.45               43.989803  71.710526
    P4  0.618421         0.660606      0.576520  0.588172              6650.74                    3.43     6654.17               43.777434  72.368421


### Reading guide

- `P0` is the baseline. It shows the minimum benchmark bar.
- `P1` captures pure Bayesian belief with no cost sensitivity.
- `P2` adds decision-theoretic expected-cost evaluation using only free evidence.
- `P3` is the full value-of-information policy, selecting paid checks by expected information gain per dollar.
- `P4` is the true decision-theoretic VOI policy, comparing expected loss now against expected loss after each evidence outcome and the evidence cost.

The experiments are intentionally separated by policy so each decision rule can be inspected independently and compared transparently.


## Failure analysis

This section extracts every missed decision for policies `P1`–`P3`, highlights the five most expensive failures, and writes dry-run notes for the top five failures in `P0`–`P4`.


In [1]:
from pathlib import Path

FAILURE_DIR = Path("experiments/failures")
FAILURE_DIR.mkdir(exist_ok=True, parents=True)


def expected_action_costs(posterior):
    return {action: sum(posterior[state] * ACTION_COSTS[action][state] for state in STATES) for action in ACTION_ORDER}


def simulate_policy(policy_name, row):
    posterior = PRIORS.copy()
    used = []
    observed = {}
    trace = []

    if policy_name == "P0":
        prediction = majority_action
        action_costs = expected_action_costs(posterior)
        decision_cost = ACTION_COSTS[prediction][row["ground_truth"]]
        return {
            "policy": policy_name,
            "test_id": row["test_id"],
            "ground_truth": row["ground_truth"],
            "ground_action": row["ground_action"],
            "prediction": prediction,
            "decision_cost": decision_cost,
            "used": used,
            "observed_evidence": observed,
            "final_posterior": posterior,
            "action_costs": action_costs,
            "trace": trace,
            "priors": PRIORS.copy(),
        }

    for evidence_key in ["E1", "E2"]:
        outcome = row.get(f"{evidence_key}_outcome")
        if outcome is None:
            continue
        prior = posterior.copy()
        posterior = bayes_update(posterior, evidence_key, outcome)
        observed[evidence_key] = outcome
        used.append(evidence_key)
        trace.append({
            "kind": "posterior_update",
            "evidence": evidence_key,
            "outcome": outcome,
            "prior": prior,
            "posterior": posterior.copy(),
        })

    if policy_name == "P1":
        best_state = max(posterior, key=posterior.get)
        prediction = state_to_action(best_state)
    elif policy_name == "P2":
        prediction, _ = choose_expected_cost_action(posterior)
    else:
        while True:
            pending = [k for k in ["E3", "E4"] if k not in used and row.get(f"{k}_outcome") is not None]
            if not pending:
                break

            if policy_name == "P3":
                ranked = []
                for evidence_key in pending:
                    ig = expected_information_gain(posterior, evidence_key)
                    score = ig / EVIDENCE_COSTS[evidence_key]
                    ranked.append({
                        "evidence": evidence_key,
                        "score": score,
                        "ig": ig,
                        "cost": EVIDENCE_COSTS[evidence_key],
                        "threshold": 0.05,
                    })
                ranked.sort(key=lambda item: item["score"], reverse=True)
                best = ranked[0]
                trace.append({
                    "kind": "entropy_gain_check",
                    "evidence": best["evidence"],
                    "score": best["score"],
                    "ig": best["ig"],
                    "cost": best["cost"],
                    "threshold": best["threshold"],
                    "comparison": "score <= threshold => stop",
                })
                if best["score"] <= 0.05:
                    break
                outcome = row.get(f"{best['evidence']}_outcome")
                if outcome is None:
                    break
                prior = posterior.copy()
                posterior = bayes_update(posterior, best["evidence"], outcome)
                observed[best["evidence"]] = outcome
                used.append(best["evidence"])
                trace.append({
                    "kind": "posterior_update",
                    "evidence": best["evidence"],
                    "outcome": outcome,
                    "prior": prior,
                    "posterior": posterior.copy(),
                })
                action, _ = choose_expected_cost_action(posterior)
                if action != "Escalate" and max(posterior.values()) >= 0.55:
                    break
            elif policy_name == "P4":
                current_action, current_costs = choose_expected_cost_action(posterior)
                current_expected_loss = min(current_costs.values())
                best_voi = None
                best_key = None
                best_details = None

                for evidence_key in pending:
                    outcomes = sorted({outcome for state in STATES for outcome in LIKELIHOODS[evidence_key][state].keys()})
                    expected_loss_after_evidence = 0.0
                    outcome_details = []
                    for outcome in outcomes:
                        p_outcome = sum(
                            posterior[state] * LIKELIHOODS[evidence_key][state].get(outcome, 0.0)
                            for state in STATES
                        )
                        if p_outcome <= 0:
                            continue
                        post = {}
                        for state in STATES:
                            likelihood = LIKELIHOODS[evidence_key][state].get(outcome, 0.0)
                            post[state] = (posterior[state] * likelihood) / p_outcome
                        action_i, costs_i = choose_expected_cost_action(post)
                        outcome_loss = min(costs_i.values())
                        expected_loss_after_evidence += p_outcome * outcome_loss
                        outcome_details.append({
                            "outcome": outcome,
                            "probability": p_outcome,
                            "posterior_after": post,
                            "chosen_action": action_i,
                            "outcome_loss": outcome_loss,
                        })
                    expected_loss_after_evidence += EVIDENCE_COSTS[evidence_key]
                    voi = current_expected_loss - expected_loss_after_evidence
                    candidate = {
                        "evidence": evidence_key,
                        "voi": voi,
                        "expected_loss_after_evidence": expected_loss_after_evidence,
                        "current_expected_loss": current_expected_loss,
                        "outcome_details": outcome_details,
                    }
                    if best_voi is None or voi > best_voi:
                        best_voi = voi
                        best_key = evidence_key
                        best_details = candidate
                if best_key is None or best_voi <= 0:
                    break
                trace.append({
                    "kind": "decision_voI",
                    "evidence": best_key,
                    "voi": best_voi,
                    "current_expected_loss": current_expected_loss,
                    "expected_loss_after_evidence": best_details["expected_loss_after_evidence"],
                    "outcome_details": best_details["outcome_details"],
                })
                outcome = row.get(f"{best_key}_outcome")
                if outcome is None:
                    break
                prior = posterior.copy()
                posterior = bayes_update(posterior, best_key, outcome)
                observed[best_key] = outcome
                used.append(best_key)
                trace.append({
                    "kind": "posterior_update",
                    "evidence": best_key,
                    "outcome": outcome,
                    "prior": prior,
                    "posterior": posterior.copy(),
                })

        if policy_name == "P3":
            prediction, _ = choose_expected_cost_action(posterior)
        elif policy_name == "P4":
            prediction, _ = choose_expected_cost_action(posterior)
        else:
            prediction = "Escalate"

    action_costs = expected_action_costs(posterior)
    decision_cost = ACTION_COSTS[prediction][row["ground_truth"]]
    return {
        "policy": policy_name,
        "test_id": row["test_id"],
        "ground_truth": row["ground_truth"],
        "ground_action": row["ground_action"],
        "prediction": prediction,
        "decision_cost": decision_cost,
        "used": used,
        "observed_evidence": observed,
        "final_posterior": posterior,
        "action_costs": action_costs,
        "trace": trace,
        "priors": PRIORS.copy(),
    }


def format_posterior(posterior):
    return "\n".join(f"  {state}: {posterior[state]:.6f}" for state in STATES)


def failure_dataframe_for_policy(policy_name):
    rows = []
    for _, row in eval_df.iterrows():
        rec = simulate_policy(policy_name, row)
        if rec["prediction"] != row["ground_action"]:
            rows.append({
                "policy": policy_name,
                "test_id": rec["test_id"],
                "ground_truth": rec["ground_truth"],
                "ground_action": rec["ground_action"],
                "prediction": rec["prediction"],
                "decision_cost": rec["decision_cost"],
                "used": rec["used"],
                "observed_evidence": rec["observed_evidence"],
                "final_posterior": rec["final_posterior"],
                "action_costs": rec["action_costs"],
                "trace": rec["trace"],
                "priors": rec["priors"],
            })
    return pd.DataFrame(rows).sort_values("decision_cost", ascending=False).reset_index(drop=True)


p0_failures_df = failure_dataframe_for_policy("P0")
p1_failures_df = failure_dataframe_for_policy("P1")
p2_failures_df = failure_dataframe_for_policy("P2")
p3_failures_df = failure_dataframe_for_policy("P3")
p4_failures_df = failure_dataframe_for_policy("P4")

wrong_decisions_df = pd.concat(
    [p1_failures_df, p2_failures_df, p3_failures_df],
    ignore_index=True,
).sort_values("decision_cost", ascending=False).reset_index(drop=True)

top_5_wrong_decisions = wrong_decisions_df.head(5).copy()


def render_failure_report(policy_name, df):
    if df.empty:
        return f"# {policy_name} failure analysis\n\nNo failures for this policy.\n"

    lines = [f"# {policy_name} failure analysis", "", f"Top {min(5, len(df))} highest-cost failures.", ""]
    for idx, row in df.head(5).iterrows():
        lines.extend([
            f"## {idx + 1}. Test case: {row['test_id']}",
            f"- True state: {row['ground_truth']}",
            f"- Ground action: {row['ground_action']}",
            f"- Predicted action: {row['prediction']}",
            f"- Decision cost paid: ${row['decision_cost']:.2f}",
            "- Evidence observed: " + (
                ", ".join(f"{k}={v}" for k, v in row["observed_evidence"].items())
                if row["observed_evidence"]
                else "none"
            ),
            "",
            "### Priors",
            "```text",
            format_posterior(row["priors"]),
            "```",
            "",
            "### Dry run / trace",
        ])
        for step in row["trace"]:
            kind = step.get("kind")
            if kind == "posterior_update":
                lines.extend([
                    f"- Evidence received: {step['evidence']} = {step['outcome']}",
                    "- Posterior calculation:",
                    "  prior -> posterior",
                    "  ```text",
                    format_posterior(step["prior"]),
                    "  ```",
                    "  ```text",
                    format_posterior(step["posterior"]),
                    "  ```",
                ])
            elif kind == "entropy_gain_check":
                lines.extend([
                    f"- Entropy-based expected information gain check for {step['evidence']}",
                    f"  IG = {step['ig']:.6f}",
                    f"  score = IG / cost = {step['score']:.6f}",
                    f"  threshold = {step['threshold']:.6f}",
                    f"  policy rule: if score <= threshold -> stop gathering evidence",
                ])
            elif kind == "decision_voI":
                lines.extend([
                    f"- Decision-theoretic VOI check for {step['evidence']}",
                    f"  current_expected_loss = {step['current_expected_loss']:.6f}",
                    f"  expected_loss_after_evidence = {step['expected_loss_after_evidence']:.6f}",
                    f"  VOI = {step['voi']:.6f}",
                    "  outcome-level expected cost calculations:",
                ])
                for outcome_detail in step["outcome_details"]:
                    lines.append(
                        f"  - Outcome {outcome_detail['outcome']} -> P = {outcome_detail['probability']:.6f}, chosen action = {outcome_detail['chosen_action']}, outcome loss = {outcome_detail['outcome_loss']:.6f}"
                    )
        lines.extend([
            "",
            "### Final posterior",
            "```text",
            format_posterior(row["final_posterior"]),
            "```",
            "",
            "### Expected cost of each action at the final posterior",
            "```text",
        ])
        for action, cost in row["action_costs"].items():
            lines.append(f"  {action}: {cost:.6f}")
        lines.extend([
            "```",
            "",
            f"### Final action: {row['prediction']}",
            "",
        ])
    return "\n".join(lines) + "\n"


for policy_name, df in {
    "P0": p0_failures_df,
    "P1": p1_failures_df,
    "P2": p2_failures_df,
    "P3": p3_failures_df,
    "P4": p4_failures_df,
}.items():
    failure_path = FAILURE_DIR / f"{policy_name.lower()}_failure.md"
    failure_path.write_text(render_failure_report(policy_name, df), encoding="utf-8")

wrong_decisions_df

top_5_wrong_decisions
